In [195]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import StandardScaler
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import functools
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [196]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/blythedineen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/blythedineen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

##  1.a. Download the datat

(1) There are 7613 training and 3263 test data points.

(2) 42.97% of the training tweets are of real disasters and 57.03% are not.

In [197]:
pd.set_option('display.max_columns', None)

In [339]:
test = pd.read_csv("nlp-getting-started/test.csv")
train = pd.read_csv("nlp-getting-started/train.csv")

In [199]:
train.shape

(7613, 5)

In [200]:
test.shape

(3263, 4)

In [201]:
target = train["target"].value_counts()

In [202]:
#target

In [203]:
print(target[1] / len(train))

0.4296597924602653


In [204]:
print(target[0] / len(train))

0.5703402075397347


##  1.b. Split the training data.

In [205]:
train_set, dev_set = train_test_split(train, test_size=0.3, random_state=42)

In [206]:
train_set.shape

(5329, 5)

In [207]:
dev_set.shape

(2284, 5)

In [208]:
train_set.head(10)

,id,keyword,location,text,target
1186,1707,bridge%20collapse,NaN,Ashes 2015: AustraliaÛªs collapse at Trent Br...,0
4071,5789,hail,"Carol Stream, Illinois",GREAT MICHIGAN TECHNIQUE CAMP\nB1G THANKS TO @...,1
5461,7789,police,Houston,CNN: Tennessee movie theater shooting suspect ...,1
5787,8257,rioting,NaN,Still rioting in a couple of hours left until ...,1
7445,10656,wounds,Lake Highlands,Crack in the path where I wiped out this morni...,0
151,216,airplane%20accident,Somewhere Out There,Experts in France begin examining airplane deb...,1
915,1324,bloody,Isolated City In World Perth,'I came to kill Indians...for FUN': Video of s...,1
1305,1886,burning,NaN,@JohnsonTionne except idk them?? it's really b...,0
2570,3685,destroy,he/him or she/her (ask),destroy the house,0
7399,10587,wounded,"Maracay y Nirgua, Venezuela",Police Officer Wounded Suspect Dead After Exch...,1


##  1.c. Pre-Processing

1. Convert all words to lowercase - stops the model from creating artificial differences between Word and word, this standardizes the text as well for any inconsistent capitalization or tweets that may have been written in all caps, but functionally the words carry the same meaning. This will make our ultimate bag of words smaller (ex. by converting word, Word, and WORD all into 1 data point word) and make the modeling slightly simplier.

2. Lemmatize all words - reduces the number of words we will be working with and groups together similar (i.e. same root) words to reduce the dimensionality of our dataset. This again will make our ultimate bag of words smaller while still representing the same concepts and groups together similar words without creating artificial division.
  
3. Strip punctuation, @, and URLs - By stripping these special characters and URLs, we are limiting our analysis to the actual words, as opposed to punctuation, special characters, and tags which are more user/tweet specific as opposed to specific to a topic.

4. Strip stopwords - By stripping stopwords, we remove the most common words in the english language that are not representative of any category or topic. For example, without removing stopwords, the most frequent words in the data set would be and, in, the... which would be overrepresented in the dataset without providing any insight into what category and would skew the dataset.

In [209]:
train_set['text'] = train_set['text'].str.lower()

In [210]:
ps = PorterStemmer()

for index,row in train_set.iterrows():
    words = word_tokenize(row['text'])
    train_set.loc[index, 'text'] = functools.reduce(lambda x, y: x + " " + ps.stem(y), words, "")

In [211]:
#strip URLs
url_pattern = r'https?://\S+|www\.\S+'
train_set['text'] = train_set['text'].str.replace(url_pattern, '', regex=True)

In [212]:
#strip special characters
train_set['text'] = train_set['text'].str.replace('/', ' ') #replace / so that cool/weird turns into cool weird instead of coolweird
train_set['text'] = train_set['text'].str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
train_set['text'] = train_set['text'].str.replace('amp', '', regex=True) #strip when ampersand has been messed up

In [213]:
stop = stopwords.words('english')
train_set['text'] = train_set['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))


In [214]:
#train_set.head(10)

##  1.d. Bag of words model


m = 2 -> X = (5329, 4277) 

m = 3 -> X = (5329, 2915) 

m = 4 -> X = (5329, 2276)

m = 5 -> X = (5329, 1868)

m = 6 -> X = (5329, 1604)

m = 7 -> X = (5329, 1416)

m = 8 -> X = (5329, 1261)

m = 9 -> X = (5329, 1138)

m = 10 -> X = (5329, 1025)

I chose an m of 5 for two reasons: first, it seemed to be approximately my elbow (where the the shape of the dataframe stops reducing at an exponential rate and starts to reduce at a much smaller rate), secondly, based on the resulting data shape, it seemed like intuitively a good balance of not overly rare words (words that only appear once or twice are intuitively noisy) while not eliminating useful data (intuitively, if a word appears 5 times or more, that feels frequent enough to be statistically significant, so choosing an m higher than 5 feels overly general and removes specificity from the data).

X shape (5329, 1868)
dev_X shape (2284, 1868)

In [215]:
m = 5

In [216]:
vectorizer = CountVectorizer(binary=True,min_df =m)
vectorizer.fit(train_set['text'])
X = vectorizer.transform(train_set['text'])

In [217]:
X.shape

(5329, 1885)

Apply data cleaning to development set

In [218]:
dev_set['text'] = dev_set['text'].str.lower()
for index,row in dev_set.iterrows():
    words = word_tokenize(row['text'])
    dev_set.loc[index, 'text'] = functools.reduce(lambda x, y: x + " " + ps.stem(y), words, "")
#strip URLs
url_pattern = r'https?://\S+|www\.\S+'
dev_set['text'] = dev_set['text'].str.replace(url_pattern, '', regex=True)
#strip special characters
dev_set['text'] = dev_set['text'].str.replace('/', ' ') #replace / so that cool/weird turns into cool weird instead of coolweird
dev_set['text'] = dev_set['text'].str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
dev_set['text'] = dev_set['text'].str.replace('amp', '', regex=True) #strip when ampersand has been messed up
dev_set['text'] = dev_set['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

In [219]:
dev_X = vectorizer.transform(dev_set['text'])

In [220]:
dev_X.shape

(2284, 1885)

## 1.e. Logistic regression

1.e.i. F1 score train: 0.9595, F1 score dev: 0.6849

1.e.ii. F1 score train: 0.8371, F1 score dev: 0.7496

1.e.iii. F1 score train: 0.8655, F1 score dev: 0.7498

1.e.iv. No regularization performed best on the training set but worst on the development set, which makes sense because it is overfitting without the regularization terms to stop overfitting and performing too well on the train and poorly on the development set. L2 regularization performed very marginally better than L1 regularization on the development set, so both are roughly performing the same as each other on development, and far better than no regularization. Our performance on the training set is slightly better for L2 than L1 but worse for both than no regularization. This shows that regularization is improving our overfitting because when we implement both L1 and L2, we are doing worse on the training but better on the development, meaning we are generalizing better.

1.e.v. The 5 most important words for deciding it is not a disaster are {heavenli, ticket, ebay, hurt, tribal} and the 5 most important to decide it is are {migrant, mh370, typhoon, hiroshima, spill}. The 5 most important words to decide it is make sense because they are intuitively connected to some of the largest disasters in the past century. The next 5 most important words are {worth, drought, wildfir, debri, earthquak}, which also intuitively correlate to natural disasters.

In [221]:
## 1.e.i. train logistic regression model without regularization terms
y = train_set['target']
clf_none = LogisticRegression(random_state=0, penalty=None).fit(X, y)
y_pred = clf_none.predict(X)
f1_score(y,y_pred)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9594920078826363

In [222]:
dev_y = dev_set['target']
dev_y_pred = clf_none.predict(dev_X)
f1_score(dev_y,dev_y_pred)

0.6849451645064806

In [223]:
## 1.e.ii. train logistic regression model with L1 regularization
clf_l1 = LogisticRegression(random_state=0, penalty='l1', solver='liblinear').fit(X, y)
y_pred = clf_l1.predict(X)
f1_score(y,y_pred)

0.8370592302373819

In [224]:
dev_y_pred = clf_l1.predict(dev_X)
f1_score(dev_y,dev_y_pred)

0.7495926127104834

In [225]:
## 1.e.iii. train logistic regression model with L2 regularization
clf_l2 = LogisticRegression(random_state=0, penalty='l2').fit(X, y)
y_pred = clf_l2.predict(X)
f1_score(y,y_pred)

0.8654595461838185

In [226]:
dev_y_pred = clf_l2.predict(dev_X)
f1_score(dev_y,dev_y_pred)

0.7498625618471688

In [248]:
## 1.e.v.
coefs = clf_l1.coef_
coefs = pd.DataFrame(coefs, columns = vectorizer.get_feature_names_out())

In [249]:
coefs = coefs.loc[:, (coefs != 0).any(axis=0)]

In [250]:
coefs.shape

(1, 786)

In [251]:
coefs = coefs.T
coefs = coefs.sort_values(by=0)

In [252]:
coefs

,0
heavenli,-4.262106
ticket,-2.496127
ebay,-2.495926
hurt,-2.254165
tribal,-2.138362
...,...
migrant,3.136937
mh370,3.143247
typhoon,3.234630
hiroshima,3.571656


In [253]:
coefs.tail(10)

,0
worth,2.624858
drought,2.664169
wildfir,2.832882
debri,2.865844
earthquak,3.036557
migrant,3.136937
mh370,3.143247
typhoon,3.234630
hiroshima,3.571656
spill,3.632340


## 1.f. Bernoulli Naive Bayes

In [265]:
X_train_BN = X.toarray()
n = X_train_BN.shape[0] # size of the dataset
d = X_train_BN.shape[1] # number of features in our dataset
K = 2

# these are the shapes of the parameters
psis = np.zeros([K,d])
phis = np.zeros([K])

# we now compute the parameters
for k in range(K):
    X_k = X_train_BN[y == k]
    psis[k] = np.mean(X_k, axis=0)
    phis[k] = X_k.shape[0] / float(n)

# print out the class proportions
print(phis)

[0.56746106 0.43253894]


In [266]:
def nb_predictions(x, psis, phis):
    """This returns class assignments and scores under the NB model.
    
    We compute \arg\max_y p(y|x) as \arg\max_y p(x|y)p(y)
    """
    # adjust shapes
    n, d = x.shape
    x = np.reshape(x, (1, n, d))
    psis = np.reshape(psis, (K, 1, d))
    
    psis = psis.clip(1e-14, 1-1e-14) # clip probabilities to avoid log(0)
    
    # compute log-probabilities
    logpy = np.log(phis).reshape([K,1])
    logpxy = x * np.log(psis) + (1-x) * np.log(1-psis)
    logpyx = logpxy.sum(axis=2) + logpy

    return logpyx.argmax(axis=0).flatten(), logpyx.reshape([K,n])

idx_train, logpyx = nb_predictions(X_train_BN, psis, phis)
print(idx_train[:10])

[1 0 1 0 1 1 1 0 0 1]


In [267]:
(idx_train==y).mean()


np.float64(0.8339275661474949)

In [270]:
#implement on dev set

idx_train, logpyx = nb_predictions(dev_X.toarray(), psis, phis)
(idx_train==dev_y).mean()

np.float64(0.8069176882661997)

## 1.g. Model comparison

Across all forms of regularization, Logistic Regression (LR) performed better on the training set (95.9\%, 83.7\%, 86.5\% accuracy for no regularization, L1, and L2 respectively) over Bernoulli Naive Bayes (BNB) (83.3\% accuracy). However, BNB performed the best on the development set (68.5\%, 74.9\%, 74.9\% accuracy for LR with(out) regularization vs 80.7\% accuracy for BNB). This means that BNB is performing more consistently on unseen data than LR, meaning the model in its current state is a better fit. Thus we can conclude while LR performed better on known data, BNB is better at predicting on unseen data so is better at predicting if a previously unseen tweet is a real disaster. This makes sense given BNB is a generative model, so it is better at handling missing data because it looks at the underlying probability distribution and can fill in missing values better, leading to less overfitting. However, a con of generative models is that they are typically less accurate for classification (which we see here at least in the training accuracy and would likely continue to observe as we better tune our two models) and it is far more complex and computationally expensive, since it requires modeling the full distribution. \\
Discriminative models like LR are typically better for supervised learning situations like this (we have data that we have already classified into our two categories that we are providing for the model) where we are labeling based on known labels and are not trying to generate new data points. In this case, if we fix the overfitting of the LR models, then we will likely find that the accuracy on classification is better for our discriminative model LR than our generative model BNB since discriminative models are better suited to this kind of task. However, we are clearly noticing the cons of discriminative models here, in that they are bad at handling missing data and tend to overfit in small or noisy datasets as we have here. Thus, in conclusion, for preliminary analysis BNB is performing better but we would find with tuning, larger datasets, and further development, over long term LR would likely be better suited to this task as we fix the overfitting and increase the dataset since generative models tend to be worse at classification once we overcome the tendency to overfit with discriminative models.

## 1.h. N-gram model

m = 1 -> X = (5329, 39925)
m = 2 -> X = (5329, 4288)
m = 3 -> X = (5329, 1871)
m = 4 -> X = (5329, 1133)
m = 5 -> X = (5329, 777)
In this case, choosing m=2 because m=1 is far too large (39925 features) but m=3 gives us 1871 features, which is essentially the same size as what we had in the previous step, so we're essentially losing all the added specificity of our ngram. Therefore, I chose m=2 because that gives us 4288 features, which is added specificity from our previous iteration without becoming overly complex and noisy like m=1.

Now implementing Logistic Regression, I implemented L2 regularization because that was the most successful before, and I got 76.6\% accuracy on the training and 61.8\% on the development set. This is worse than what we achieved before. Now looking at BNB, I got 82.1\% accuracy on the training and  73.6\% accuracy on the test. This is also worse than before. This implies that in this case, n-grams do not provide further information than what we were able to get by words on their own, and instead it just adds noise that makes the predictions worse in both cases. This means that n-grams are not a useful feature for this particular modeling case.

In [305]:
ngram_m = 2

In [310]:
ngram_vect = CountVectorizer(binary=True,min_df =ngram_m,ngram_range=(2,2))
ngram_vect.fit(train_set['text'])
ngram_X = ngram_vect.transform(train_set['text'])
ngram_dev_X = ngram_vect.transform(dev_set['text'])

In [311]:
ngram_X.shape

(5329, 4288)

In [312]:
#train logistic regression model with L2 (best performance)
clf_l2_ngram = LogisticRegression(random_state=0, penalty='l2').fit(ngram_X, y)
y_pred = clf_l2_ngram.predict(ngram_X)
f1_score(y,y_pred)

0.766287487073423

In [316]:
dev_y_pred = clf_l2_ngram.predict(ngram_dev_X)
f1_score(dev_y,dev_y_pred)

0.6178967994774657

In [317]:
X_train_BN = ngram_X.toarray()
n = X_train_BN.shape[0] # size of the dataset
d = X_train_BN.shape[1] # number of features in our dataset
K = 2

# these are the shapes of the parameters
psis = np.zeros([K,d])
phis = np.zeros([K])

# we now compute the parameters
for k in range(K):
    X_k = X_train_BN[y == k]
    psis[k] = np.mean(X_k, axis=0)
    phis[k] = X_k.shape[0] / float(n)

# print out the class proportions
print(phis)

[0.56746106 0.43253894]


In [318]:
idx_train, logpyx = nb_predictions(X_train_BN, psis, phis)
(idx_train==y).mean()

np.float64(0.8211671983486583)

In [320]:
#implement on dev set
idx_train, logpyx = nb_predictions(ngram_dev_X.toarray(), psis, phis)
(idx_train==dev_y).mean()

np.float64(0.7364273204903677)

## 1.i. Determine performance with the test set

I implemented regular CountVectorizer with no n-grams since I found it to be less effective in (h) and I implemented Logistic Regression with L2 regularization because it was the best performing of my discriminative models. I chose to use a discriminative model over a generative, even though in testing Bernoulli Naive Bayes performed better, because discriminative models tend to perform better for classification of supervised data than generative. In training, my new classifier had an accuracy of 86.7\%. After submitting to Kaggle, I got an accuracy score of 0.78547. While lower than what I saw in training, this was about what I expected. We were working with a relatively small data set so I was not anticipating an incredibly accurate result. I find I am pleasantly surprised by that outcome, and while it is far from good. That being said, based on the accuracy we found during the previous steps, this was lower than would be expected based on the higher accuracy we were able to find with our holdout set.

In [330]:
y = train["target"]

In [331]:
##apply data cleaning to whole train set
train['text'] = train['text'].str.lower()
for index,row in train.iterrows():
    words = word_tokenize(row['text'])
    train.loc[index, 'text'] = functools.reduce(lambda x, y: x + " " + ps.stem(y), words, "")
#strip URLs
url_pattern = r'https?://\S+|www\.\S+'
train['text'] = train['text'].str.replace(url_pattern, '', regex=True)
#strip special characters
train['text'] = train['text'].str.replace('/', ' ') #replace / so that cool/weird turns into cool weird instead of coolweird
train['text'] = train['text'].str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
train['text'] = train['text'].str.replace('amp', '', regex=True) #strip when ampersand has been messed up
train['text'] = train['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

In [332]:
vectorizer = CountVectorizer(binary=True,min_df =5)
vectorizer.fit(train['text'])
X = vectorizer.transform(train['text'])

In [333]:
## train logistic regression model with L2 regularization
clf_l2 = LogisticRegression(random_state=0, penalty='l2').fit(X, y)
y_pred = clf_l2.predict(X)
f1_score(y,y_pred)

0.867053888351081

In [334]:
##apply data cleaning to whole test set
test['text'] = test['text'].str.lower()
for index,row in test.iterrows():
    words = word_tokenize(row['text'])
    test.loc[index, 'text'] = functools.reduce(lambda x, y: x + " " + ps.stem(y), words, "")
#strip URLs
url_pattern = r'https?://\S+|www\.\S+'
test['text'] = test['text'].str.replace(url_pattern, '', regex=True)
#strip special characters
test['text'] = test['text'].str.replace('/', ' ') #replace / so that cool/weird turns into cool weird instead of coolweird
test['text'] = test['text'].str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
test['text'] = test['text'].str.replace('amp', '', regex=True) #strip when ampersand has been messed up
test['text'] = test['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

In [335]:
#vectorize test set
X_test = vectorizer.transform(test['text'])

In [336]:
y_pred_test = clf_l2.predict(X_test)

In [344]:
y_pred_test

array([1, 1, 1, ..., 1, 1, 0])

In [342]:
ids = pd.DataFrame(test['id'])

In [345]:
csv = pd.DataFrame(y_pred_test)
csv.columns = ['target']
csv = ids.join(csv)

In [346]:
csv

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
...,...,...
3258,10861,0
3259,10865,1
3260,10868,1
3261,10874,1


In [347]:
csv.to_csv('y_pred.csv', index=False)